# 01 — Data Acquisition

This notebook downloads and caches all datasets needed for the Autobahn speed safety analysis.

**Datasets:**
1. 🇩🇪 **Unfallatlas** — GPS-level individual accident records (Germany, 2016–2023)
2. 🇩🇪 **Destatis GENESIS** — Aggregated accident stats by road type (Germany)
3. 🇳🇱 **CBS OData** — Road accident casualties by road type (Netherlands)
4. 🇳🇱 **BRON** — Individual accident records (Netherlands) — manual download
5. 🇪🇺 **ERSO/CARE** — EU comparative road safety statistics

Run cells in order. All data is saved to `../data/raw/` (gitignored).

---

**Prerequisites:**
- Copy `.env.example` to `.env` and fill in Destatis credentials (free registration at https://www-genesis.destatis.de/)
- `uv sync` to install dependencies

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from autobahn_safety.data_loaders import (
    download_unfallatlas,
    load_unfallatlas,
    download_destatis_timeseries,
    load_destatis_autobahn,
    fetch_cbs_odata,
)

load_dotenv()

DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_RAW.resolve()}")


Data directory: /Users/dennis/repos/autobahn-speed-safety-analysis/data/raw


## 1. Germany — Unfallatlas

Individual GPS-located accident records for all of Germany, 2016–2023.  
Published jointly by the Statistische Landesämter.

**Source:** https://unfallatlas.statistikportal.de/  
**Direct URL pattern:** `https://www.opengeodata.nrw.de/produkte/transport_verkehr/unfallatlas/Unfallorte{YEAR}_EPSG25832_CSV.zip`

Key fields:
- `UKATEGORIE`: 1=fatal, 2=serious injury, 3=slight injury
- `INN_ORT`: 0 = outside built-up area (includes Autobahn)
- `XGCSWGS84`, `YGCSWGS84`: WGS84 coordinates

In [2]:
UNFALLATLAS_DIR = DATA_RAW / "germany" / "unfallatlas"
YEARS = list(range(2016, 2025))  # 2016–2024

for year in YEARS:
    year_dir = UNFALLATLAS_DIR / str(year)
    try:
        download_unfallatlas(year, year_dir)
    except Exception as e:
        print(f"{year}: FAILED — {e}")


Unfallatlas 2016: 100%|██████████| 7.04M/7.04M [00:00<00:00, 23.1MB/s]


2016: done


Unfallatlas 2017: 100%|██████████| 8.20M/8.20M [00:00<00:00, 22.1MB/s]


2017: done


Unfallatlas 2018: 100%|██████████| 7.70M/7.70M [00:00<00:00, 18.4MB/s]


...

2021: done


Unfallatlas 2022: 100%|██████████| 13.3M/13.3M [00:00<00:00, 24.4MB/s]


2022: done


Unfallatlas 2023: 100%|██████████| 12.7M/12.7M [00:00<00:00, 20.8MB/s]


2023: done


In [3]:
# Load all years into a single DataFrame
df_unfallatlas = load_unfallatlas(UNFALLATLAS_DIR, YEARS)
print(f"Loaded {len(df_unfallatlas):,} accident records")
print(f"Columns: {list(df_unfallatlas.columns)}")
df_unfallatlas.head()

Loaded 256,492 accident records
Columns: ['OBJECTID', 'UIDENTSTLAE', 'ULAND', 'UREGBEZ', 'UKREIS', 'UGEMEINDE', 'UJAHR', 'UMONAT', 'USTUNDE', 'UWOCHENTAG', 'UKATEGORIE', 'UART', 'UTYP1', 'ULICHTVERH', 'IstStrassenzustand', 'IstRad', 'IstPKW', 'IstFuss', 'IstKrad', 'IstGkfz', 'IstSonstige', 'LINREFX', 'LINREFY', 'XGCSWGS84', 'YGCSWGS84', 'year']


,OBJECTID,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,...,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,XGCSWGS84,YGCSWGS84,year
0,1,01220204125013262022,1,0,54,84,2022,2,19,6,...,1,0,0,0,0,"506085,644000001018867","6035085,351999809965491","9,093886180000030","54,463395769999998",2022
1,2,01220529134013152022,1,0,57,44,2022,5,11,1,...,0,0,1,0,0,"593821,621226734016091","6014331,532445689663291","10,440636083000101","54,268303975000002",2022
2,3,01220508125013982022,1,0,59,73,2022,5,12,1,...,1,0,0,0,0,"540417,076000001979992","6045563,334999830462039","9,624949321000029","54,555985573000001",2022
3,4,01220517152013752022,1,0,3,0,2022,5,8,3,...,0,0,0,0,0,"609966,338422868051566","5970404,848156260326505","10,672490491000101","53,870453103000003",2022
4,5,01220426181013142022,1,0,61,46,2022,4,19,3,...,0,0,0,0,0,"533425,841011611977592","5975832,250405009835958","9,509078523000030","53,929808971999996",2022


## 2. Germany — Destatis GENESIS-Online

Aggregated accident statistics by road type (Autobahn, Bundesstraße, etc.) and severity.  
Useful for long-term trends and normalization with official vehicle-km figures.

**Requires:** Free account at https://www-genesis.destatis.de/  
**Set in `.env`:** `DESTATIS_USERNAME` and `DESTATIS_PASSWORD`

**Key tables:**
- `46241-0023` — accidents by category and location type
- `46241-0031` — accidents on motorways (Autobahn) specifically

In [ ]:
DESTATIS_DIR = DATA_RAW / "germany" / "destatis"
DESTATIS_DIR.mkdir(parents=True, exist_ok=True)

from autobahn_safety.data_loaders import download_destatis_timeseries, load_destatis_autobahn

xlsx_path = download_destatis_timeseries(DESTATIS_DIR)

# Preview: Autobahn accidents time series
df_autobahn = load_destatis_autobahn(xlsx_path)
print(f"Loaded {len(df_autobahn)} years of Autobahn accident data")
df_autobahn.tail(10)


## 3. Netherlands — CBS OData API

Road accident casualty statistics by road type, available via CBS (Statistics Netherlands) open API.  
No registration required.

**Portal:** https://opendata.cbs.nl/  
**API base:** `https://opendata.cbs.nl/ODataApi/odata/{DATASET_ID}/`

Search CBS StatLine for:
- *verkeersslachtoffers* (traffic casualties)
- *verkeersongevallen* (traffic accidents)
- Filter by road type: *autosnelweg* = motorway

In [ ]:
CBS_DIR = DATA_RAW / 'netherlands' / 'cbs'
CBS_DIR.mkdir(parents=True, exist_ok=True)

# 3a: Traffic deaths (overall NL totals)
for ds_id, name in [('71426ned', 'deaths_by_province'), ('71936ned', 'deaths_by_mode')]:
    out = CBS_DIR / f'{name}.csv'
    if out.exists():
        print(f'{ds_id}: already cached')
        continue
    df = fetch_cbs_odata(ds_id)
    df.to_csv(out, index=False)
    print(f'{ds_id}: {len(df)} rows saved')

# 3c: Vehicle-km by vehicle type (exposure data for rate normalization)
out_vkm = CBS_DIR / 'vehicle_km_by_type.csv'
if not out_vkm.exists():
    df_vkm = fetch_cbs_odata('85395NED')
    df_vkm.to_csv(out_vkm, index=False)
    print(f'85395NED: {len(df_vkm)} rows saved (vehicle-km)')
else:
    print('85395NED: already cached')

# 3b: BRON - manual download
BRON_DIR = DATA_RAW / 'netherlands' / 'bron'
BRON_DIR.mkdir(parents=True, exist_ok=True)
bron_files = list(BRON_DIR.glob('*.csv')) + list(BRON_DIR.glob('*.xlsx'))
if bron_files:
    print(f'BRON: found {[f.name for f in bron_files]}')
else:
    print('WARNING: BRON not found - manual download required:')
    print('  https://data.overheid.nl -> search BRON verkeersongevallen')
    print(f'  Extract to: {BRON_DIR.resolve()}')


In [ ]:
# Fetch the actual data
out_path = CBS_DIR / f"{dataset_id}.csv"

if out_path.exists():
    df_cbs = pd.read_csv(out_path)
    print(f"Loaded from cache: {len(df_cbs)} rows")
else:
    df_cbs = fetch_cbs_odata(dataset_id)
    df_cbs.to_csv(out_path, index=False)
    print(f"Fetched and saved: {len(df_cbs)} rows")

df_cbs.head()

## 4. Netherlands — BRON (Individual accident records)

BRON = *Bestand geRegistreerde Ongevallen in Nederland* — the official Dutch individual accident database.  
Maintained by CBS, SWOV, and Rijkswaterstaat.

**Download:**
1. Go to https://data.overheid.nl and search for **"BRON verkeersongevallen"**
2. Or request from SWOV: https://www.swov.nl/feiten-cijfers/bronnen-en-methoden/bron
3. Extract to `../data/raw/netherlands/bron/`

**Key fields:** road type, speed limit, severity, coordinates (RD New / WGS84), year

In [ ]:
BRON_DIR = DATA_RAW / "netherlands" / "bron"
BRON_DIR.mkdir(parents=True, exist_ok=True)

bron_files = list(BRON_DIR.glob("*.csv")) + list(BRON_DIR.glob("*.xlsx"))
if bron_files:
    print(f"Found BRON files: {[f.name for f in bron_files]}")
    # Load and inspect
    df_bron = pd.read_csv(bron_files[0]) if bron_files[0].suffix == ".csv" else pd.read_excel(bron_files[0])
    print(f"Shape: {df_bron.shape}")
    df_bron.head()
else:
    print("⚠️  No BRON data found.")
    print(f"   → Download manually and place in: {BRON_DIR.resolve()}")
    print("   → https://data.overheid.nl (search 'BRON verkeersongevallen')")

## 5. EU Comparative — ERSO / CARE Database

The European Road Safety Observatory provides standardized cross-country statistics.  
Useful for a clean DE vs NL comparison on a common baseline.

**Portal:** https://road-safety.transport.ec.europa.eu/  
**Country profiles:** DE and NL, statistics by road type, 1991–present

Manual download steps:
1. Go to the ERSO portal
2. Navigate to Statistics → Country profiles
3. Download Germany and Netherlands fatality tables
4. Save to `../data/raw/eu/erso/`

In [ ]:
ERSO_DIR = DATA_RAW / "eu" / "erso"
ERSO_DIR.mkdir(parents=True, exist_ok=True)

erso_files = list(ERSO_DIR.glob("*.csv")) + list(ERSO_DIR.glob("*.xlsx"))
if erso_files:
    print(f"Found ERSO files: {[f.name for f in erso_files]}")
else:
    print("⚠️  No ERSO data found.")
    print(f"   → Download from https://road-safety.transport.ec.europa.eu/")
    print(f"   → Place in: {ERSO_DIR.resolve()}")

## Summary

| Dataset | Status | Records | Notes |
|---------|--------|---------|-------|
| Unfallatlas (DE) | — | — | Run cells above |
| Destatis GENESIS (DE) | — | — | Needs .env credentials |
| CBS OData (NL) | — | — | Auto-fetched |
| BRON (NL) | — | — | Manual download |
| ERSO/CARE (EU) | — | — | Manual download |

→ Proceed to `02_data_exploration.ipynb` once data is downloaded.